In [4]:
import pandas as pd
import numpy as np
from itertools import permutations
import os

# =====================================================
# 0. 設定
# =====================================================

CSV_FILE = "play_info.csv"

print("現在Pythonが見ているフォルダ:")
print(os.getcwd())

print("\nこのフォルダ内のファイル:")
print(os.listdir())

# =====================================================
# 1. CSV読み込み：文字コード自動対応版
# =====================================================

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(
        f"{CSV_FILE} が見つかりません。ファイル名が違う可能性があります。"
    )

encodings = ["utf-8-sig", "utf-8", "cp932", "shift_jis"]

df = None
used_encoding = None

for enc in encodings:
    try:
        df = pd.read_csv(CSV_FILE, encoding=enc)
        used_encoding = enc
        break
    except UnicodeDecodeError:
        continue

if df is None:
    raise ValueError(
        "CSVを読み込めませんでした。utf-8-sig, utf-8, cp932, shift_jis のどれでも失敗しました。"
    )

print("\nCSVを読み込みました。")
print(f"使用した文字コード: {used_encoding}")

print("\n現在のCSV列名:")
print(df.columns.tolist())

print("\n先頭5行:")
display(df.head())

# =====================================================
# 2. 必要列の確認
# =====================================================

required_columns = [
    "player", "obp", "ops", "iso", "bb_rate",
    "k_rate", "hr", "sb", "re24", "wpa"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("\nCSVに足りない列があります:")
    print(missing_columns)

    print("\n必要な列:")
    print(required_columns)

    print("\n現在のCSV列名:")
    print(df.columns.tolist())

    raise ValueError("列名が合っていません。上に表示されたCSV列名を送ってください。")

# =====================================================
# 3. 数値変換
# =====================================================

numeric_columns = [
    "obp", "ops", "iso", "bb_rate",
    "k_rate", "hr", "sb", "re24", "wpa"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 欠損値がある場合は平均で補完
for col in numeric_columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mean())

# 選手名が欠損している行を除外
df = df.dropna(subset=["player"]).copy()

# 同じ選手名が重複していた場合は最初の1行だけ使う
df = df.drop_duplicates(subset=["player"]).copy()

if len(df) < 9:
    raise ValueError("選手が9人未満です。9人以上のデータを入れてください。")

print(f"\n使用する選手数: {len(df)}人")

# =====================================================
# 4. 指標を0〜100点に正規化
# =====================================================

def normalize_high(series):
    """
    高いほど良い指標を0〜100点に変換する
    """
    if series.max() == series.min():
        return pd.Series([50] * len(series), index=series.index)
    return (series - series.min()) / (series.max() - series.min()) * 100


def normalize_low(series):
    """
    低いほど良い指標を0〜100点に変換する
    例: K%
    """
    if series.max() == series.min():
        return pd.Series([50] * len(series), index=series.index)
    return (series.max() - series) / (series.max() - series.min()) * 100


df["obp_score"] = normalize_high(df["obp"])
df["ops_score"] = normalize_high(df["ops"])
df["iso_score"] = normalize_high(df["iso"])
df["bb_score"] = normalize_high(df["bb_rate"])
df["k_score"] = normalize_low(df["k_rate"])
df["hr_score"] = normalize_high(df["hr"])
df["sb_score"] = normalize_high(df["sb"])
df["re24_score"] = normalize_high(df["re24"])
df["wpa_score"] = normalize_high(df["wpa"])

# =====================================================
# 5. 打順ごとの評価式
# =====================================================

def calc_batting_order_scores(row):
    scores = {}

    # 1番：出塁・選球眼・足
    scores[1] = (
        row["obp_score"] * 0.30 +
        row["bb_score"] * 0.15 +
        row["k_score"] * 0.15 +
        row["sb_score"] * 0.15 +
        row["ops_score"] * 0.10 +
        row["re24_score"] * 0.10 +
        row["wpa_score"] * 0.05
    )

    # 2番：出塁・コンタクト・つなぎ
    scores[2] = (
        row["obp_score"] * 0.25 +
        row["k_score"] * 0.20 +
        row["bb_score"] * 0.15 +
        row["ops_score"] * 0.15 +
        row["re24_score"] * 0.15 +
        row["sb_score"] * 0.05 +
        row["wpa_score"] * 0.05
    )

    # 3番：総合打撃力
    scores[3] = (
        row["ops_score"] * 0.30 +
        row["obp_score"] * 0.20 +
        row["iso_score"] * 0.15 +
        row["re24_score"] * 0.20 +
        row["wpa_score"] * 0.10 +
        row["hr_score"] * 0.05
    )

    # 4番：長打力・得点期待値・勝利貢献
    scores[4] = (
        row["iso_score"] * 0.25 +
        row["hr_score"] * 0.20 +
        row["ops_score"] * 0.20 +
        row["re24_score"] * 0.20 +
        row["wpa_score"] * 0.15
    )

    # 5番：4番の後ろで返す力
    scores[5] = (
        row["ops_score"] * 0.25 +
        row["iso_score"] * 0.25 +
        row["hr_score"] * 0.15 +
        row["re24_score"] * 0.20 +
        row["wpa_score"] * 0.10 +
        row["obp_score"] * 0.05
    )

    # 6番：下位打線の中で打撃力が高い選手
    scores[6] = (
        row["ops_score"] * 0.30 +
        row["iso_score"] * 0.20 +
        row["obp_score"] * 0.15 +
        row["re24_score"] * 0.20 +
        row["hr_score"] * 0.10 +
        row["wpa_score"] * 0.05
    )

    # 7番：一発・最低限の打撃力
    scores[7] = (
        row["ops_score"] * 0.25 +
        row["iso_score"] * 0.20 +
        row["hr_score"] * 0.15 +
        row["k_score"] * 0.15 +
        row["re24_score"] * 0.15 +
        row["obp_score"] * 0.10
    )

    # 8番：出塁・三振の少なさ
    scores[8] = (
        row["obp_score"] * 0.25 +
        row["k_score"] * 0.25 +
        row["bb_score"] * 0.15 +
        row["sb_score"] * 0.10 +
        row["ops_score"] * 0.10 +
        row["re24_score"] * 0.10 +
        row["wpa_score"] * 0.05
    )

    # 9番：1番につなぐ役割
    scores[9] = (
        row["obp_score"] * 0.30 +
        row["sb_score"] * 0.20 +
        row["k_score"] * 0.20 +
        row["bb_score"] * 0.15 +
        row["ops_score"] * 0.10 +
        row["re24_score"] * 0.05
    )

    return scores

# =====================================================
# 6. 各選手の打順別スコアを作成
# =====================================================

score_rows = []

for _, row in df.iterrows():
    player_scores = calc_batting_order_scores(row)

    for order, score in player_scores.items():
        score_rows.append({
            "player": row["player"],
            "order": order,
            "score": score
        })

score_df = pd.DataFrame(score_rows)

print("\n打順別スコア表:")
display(score_df.head(20))

# =====================================================
# 7. 最適打順を決める
# =====================================================

candidate_players = set()

for order in range(1, 10):
    top_players = (
        score_df[score_df["order"] == order]
        .sort_values("score", ascending=False)
        .head(6)["player"]
        .tolist()
    )
    candidate_players.update(top_players)

candidate_players = list(candidate_players)

print("\n候補選手:")
print(candidate_players)

if len(candidate_players) < 9:
    raise ValueError("候補選手が9人未満です。CSVに9人以上の選手を入れてください。")

# 候補が多すぎると重くなるので最大10人に制限
MAX_CANDIDATES = 10
candidate_players = candidate_players[:MAX_CANDIDATES]

print(f"\n最適化に使う候補選手数: {len(candidate_players)}人")

best_lineup = None
best_score = -1

for lineup in permutations(candidate_players, 9):
    total_score = 0

    for i, player in enumerate(lineup):
        batting_order = i + 1

        player_score = score_df[
            (score_df["player"] == player) &
            (score_df["order"] == batting_order)
        ]["score"].values[0]

        total_score += player_score

    if total_score > best_score:
        best_score = total_score
        best_lineup = lineup

# =====================================================
# 8. 結果を表にする
# =====================================================

result_rows = []

for i, player in enumerate(best_lineup):
    batting_order = i + 1

    score = score_df[
        (score_df["player"] == player) &
        (score_df["order"] == batting_order)
    ]["score"].values[0]

    player_data = df[df["player"] == player].iloc[0]

    result_rows.append({
        "打順": batting_order,
        "選手名": player,
        "スコア": round(score, 2),
        "OBP": player_data["obp"],
        "OPS": player_data["ops"],
        "ISO": player_data["iso"],
        "BB%": player_data["bb_rate"],
        "K%": player_data["k_rate"],
        "HR": player_data["hr"],
        "SB": player_data["sb"],
        "RE24": player_data["re24"],
        "WPA": player_data["wpa"]
    })

result_df = pd.DataFrame(result_rows)

print("\nおすすめ打順")
print("==============")
display(result_df)

print(f"\n合計スコア: {best_score:.2f}")

# =====================================================
# 9. 理由を表示
# =====================================================

def make_reason(order, row):
    player = row["選手名"]

    if order == 1:
        return f"{player}は出塁率・選球眼・走塁面を重視した1番向きの評価が高いです。"
    elif order == 2:
        return f"{player}は出塁力と三振の少なさがあり、つなぎ役の2番に向いています。"
    elif order == 3:
        return f"{player}はOPSやRE24を含めた総合打撃力が高く、3番に向いています。"
    elif order == 4:
        return f"{player}はISO・HR・OPS・RE24・WPAを重視した中軸評価が高く、4番に向いています。"
    elif order == 5:
        return f"{player}は長打力とRE24があり、4番の後ろで走者を返す5番に向いています。"
    elif order == 6:
        return f"{player}は下位打線の中で打撃力を発揮できる6番向きの評価です。"
    elif order == 7:
        return f"{player}は一発や最低限の打撃力を期待できるため、7番に向いています。"
    elif order == 8:
        return f"{player}は出塁や三振の少なさを評価し、8番に配置されています。"
    elif order == 9:
        return f"{player}は出塁・走塁・三振の少なさを評価し、1番につなぐ9番に向いています。"

print("\n打順の理由")
print("==============")

for _, row in result_df.iterrows():
    print(f"{row['打順']}番 {row['選手名']}: {make_reason(row['打順'], row)}")

# =====================================================
# 10. 結果をCSV保存
# =====================================================

result_df.to_csv("recommended_lineup.csv", index=False, encoding="utf-8-sig")

print("\nrecommended_lineup.csv として結果を保存しました。")

現在Pythonが見ているフォルダ:
/Users/atsuki/-II

このフォルダ内のファイル:
['001.ipynb', '2023年', 'RE24 2025年', 'play_info.csv', '002.ipynb', '2024年', '.gitignore', '2025年', '中間発表に向けて', '.git']


/var/folders/7z/j354sfjn68qdc68lznr2tqdh0000gn/T/ipykernel_12948/2259469803.py:34: DtypeWarning: Columns (125,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, encoding=enc)



CSVを読み込みました。
使用した文字コード: cp932

現在のCSV列名:
['game_id', 'seqno_9', 'year_id', 'game_kind_id', 'game_kind_name', 'game_date', 'game_year', 'game_month', 'stadium_id', 'stadium_name', 'home_team_league_id', 'home_team_league_name', 'home_team_id', 'home_team_name', 'away_team_league_id', 'away_team_league_name', 'away_team_id', 'away_team_name', 'play_date', 'play_time', 'inning', 'top_bottom_id', 'top_bottom_name', 'pa_of_inning', 'np_of_pa', 'pitcher_team_league_id', 'pitcher_team_league_name', 'pitcher_team_id', 'pitcher_team_name', 'pitcher_id', 'pitcher_name', 'pitcher_handedness', 'is_starter', 'bf_of_game', 'np_of_game', 'batter_team_league_id', 'batter_team_league_name', 'batter_team_id', 'batter_team_name', 'batter_id', 'batter_name', 'batter_handedness', 'batter_pos', 'batting_order', 'is_pinch_hitter', 'pitch_result', 'pa_result', 'run_scored', 'pickoff_attempt_to', 'pitch_type_id', 'pitch_type_name', 'pitch_type_name_detail', 'pitch_type_group', 'pitch_speed', 'pitch_location_x

,game_id,seqno_9,year_id,game_kind_id,game_kind_name,game_date,game_year,game_month,stadium_id,stadium_name,...,is_hard,is_productive_out,is_sac_bunt_attempt,is_wp,is_bk,is_pb,is_pk,in_dirt,off_wall,change_situation
0,2021029038,11010101,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,0,0,0
1,2021029038,11010201,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,0,0,1
2,2021029038,11020101,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,0,0,1
3,2021029038,11030101,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,1,0,0
4,2021029038,11030201,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,0,0,0



CSVに足りない列があります:
['player', 'obp', 'ops', 'iso', 'bb_rate', 'k_rate', 'hr', 'sb', 're24', 'wpa']

必要な列:
['player', 'obp', 'ops', 'iso', 'bb_rate', 'k_rate', 'hr', 'sb', 're24', 'wpa']

現在のCSV列名:
['game_id', 'seqno_9', 'year_id', 'game_kind_id', 'game_kind_name', 'game_date', 'game_year', 'game_month', 'stadium_id', 'stadium_name', 'home_team_league_id', 'home_team_league_name', 'home_team_id', 'home_team_name', 'away_team_league_id', 'away_team_league_name', 'away_team_id', 'away_team_name', 'play_date', 'play_time', 'inning', 'top_bottom_id', 'top_bottom_name', 'pa_of_inning', 'np_of_pa', 'pitcher_team_league_id', 'pitcher_team_league_name', 'pitcher_team_id', 'pitcher_team_name', 'pitcher_id', 'pitcher_name', 'pitcher_handedness', 'is_starter', 'bf_of_game', 'np_of_game', 'batter_team_league_id', 'batter_team_league_name', 'batter_team_id', 'batter_team_name', 'batter_id', 'batter_name', 'batter_handedness', 'batter_pos', 'batting_order', 'is_pinch_hitter', 'pitch_result', 'pa_result

ValueError: 列名が合っていません。上に表示されたCSV列名を送ってください。

In [5]:
import pandas as pd
import os

CSV_FILE = "play_info.csv"

encodings = ["utf-8-sig", "utf-8", "cp932", "shift_jis"]

df = None
used_encoding = None

for enc in encodings:
    try:
        df = pd.read_csv(CSV_FILE, encoding=enc)
        used_encoding = enc
        break
    except UnicodeDecodeError:
        continue

print("使用した文字コード:", used_encoding)
print("列名一覧:")
for col in df.columns:
    print(col)

display(df.head())

/var/folders/7z/j354sfjn68qdc68lznr2tqdh0000gn/T/ipykernel_12948/4183810853.py:13: DtypeWarning: Columns (125,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, encoding=enc)


使用した文字コード: cp932
列名一覧:
game_id
seqno_9
year_id
game_kind_id
game_kind_name
game_date
game_year
game_month
stadium_id
stadium_name
home_team_league_id
home_team_league_name
home_team_id
home_team_name
away_team_league_id
away_team_league_name
away_team_id
away_team_name
play_date
play_time
inning
top_bottom_id
top_bottom_name
pa_of_inning
np_of_pa
pitcher_team_league_id
pitcher_team_league_name
pitcher_team_id
pitcher_team_name
pitcher_id
pitcher_name
pitcher_handedness
is_starter
bf_of_game
np_of_game
batter_team_league_id
batter_team_league_name
batter_team_id
batter_team_name
batter_id
batter_name
batter_handedness
batter_pos
batting_order
is_pinch_hitter
pitch_result
pa_result
run_scored
pickoff_attempt_to
pitch_type_id
pitch_type_name
pitch_type_name_detail
pitch_type_group
pitch_speed
pitch_location_x
pitch_location_y
pitch_target_location_x
pitch_target_location_y
pitch_zone
pitch_zone_side
pitch_zone_height
batted_ball_location_x
batted_ball_location_y
batted_ball_grounded_locat

,game_id,seqno_9,year_id,game_kind_id,game_kind_name,game_date,game_year,game_month,stadium_id,stadium_name,...,is_hard,is_productive_out,is_sac_bunt_attempt,is_wp,is_bk,is_pb,is_pk,in_dirt,off_wall,change_situation
0,2021029038,11010101,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,0,0,0
1,2021029038,11010201,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,0,0,1
2,2021029038,11020101,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,0,0,1
3,2021029038,11030101,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,1,0,0
4,2021029038,11030201,30022,1,セ・リーグ公式戦,2025-03-28,2025,3,1,東京ドーム,...,0,0,0,0,0,0,0,0,0,0
